In [ ]:
from entities import TravelerGroup, Traveler, System
import numpy as np
import copy

In [ ]:
from plots import plot_policy_convergence, plot_expected_value_convergence, plot_policy, plot_transition_matrix,plot_karma_level_distribution, plot_interactive_policy
from IPython.display import clear_output

In [ ]:
# -------------------------------------------------------------
# 1. Define model dimensions and parameters
# -------------------------------------------------------------
U = 2 # u in {0, 1} (urgency levels)             
T = 10 # t in {0, 1, ..., 9} (time slots)         
delta_t = 15          
n_travelers = 9000 
K = 50 # k in {0, 1, ..., 100} (karma levels)
k_init = 10 # 10 
n_groups = 1
t_star = 8
phi = np.array([[0.8, 0.2],
                [0.8, 0.2]])

u_value = np.array([1.0, 6.0]) # from the paper

# per hour penalty
delta = 0.99 # discount factor  
alpha = 0.5# 1 #0.05 # crowdedness penalty weight
beta = 4/60 # early arrival weight (per minute)
gamma = 16/60 # late arrival weight (per minute)

In [ ]:
# -------------------------------------------------------------
# 2. Create traveler groups
# -------------------------------------------------------------
groups = []

g1 = TravelerGroup(
    type_id=0,
    phi=phi,
    delta_t= delta_t,
    t_star=t_star,
    u_value=u_value,
    K=K,
    T=T,
    delta=delta,
    alpha=alpha,
    beta=beta,
    gamma=gamma
)
groups.append(g1)
# -------------------------------------------------------------
# 3. Create travelers and split across groups
# -------------------------------------------------------------
travelers = []
per_group = n_travelers // n_groups

traveler_id = 0
for group in groups:
    for _ in range(per_group):
        traveler = Traveler(group=group, k_init=k_init, id=traveler_id)
        travelers.append(traveler)
        traveler_id += 1

# -------------------------------------------------------------
# 4. Initialize the System with all travelers
# -------------------------------------------------------------
first_class_capacity = 70# 90 # 12 * delta_t # seat per class per departure time 
second_class_capacity = 630# 810 # 48 * delta_t 

system = System(
    first_class_capacity=first_class_capacity,
    second_class_capacity=second_class_capacity,
    K=K,
    T=T,
    travelers=travelers
)

In [ ]:
# -------------------------------------------------------------
# 5. Simulation loop
# -------------------------------------------------------------
threshold = 1e-4
total_day = 40#100
current_day = total_day

# For storing old policies: (states × actions × groups)
pi_old = np.zeros((U*(K+1), T*(K+1), n_groups))
error_vec = np.zeros((current_day, n_groups))
expected_value_vec = np.zeros((current_day, n_groups))
group_history = {}
b_star_history = {}

converge = False

while not converge and current_day > 0:
    print("----- Remaining day", current_day, "-----")
    
    for g in groups:
        pi_old[:, :, g.traveler_type] = g.pi.copy()

    converge = True
    current_day -= 1

    # 1. Travelers act
    for tr in travelers:
        tr.store_start_state()
        tr.action()

    # 2. System queues
    system.simulate_class_attribution()

    total_karma = 0
    # 3. Payment
    for tr in travelers:
        total_karma += tr.k_curr # for the previous day
        tr.paid_karma_bid()
        
    assert not (np.isclose(total_karma, n_travelers * k_init) == False), f"Total karma {total_karma} does not match initial total karma {n_travelers * k_init}"

    # 4. Redistribution
    system.karma_redistribution()

    # 5. Update urgency
    for tr in travelers:
        tr.update_urgency()

    # 6. Update each group (independent policies)
    for g in groups:

        g.update_group_attributes(system, (total_day-current_day)) 


        g.update_state_distribution()
        g.compute_expected_value_function()

        # check that the transition matrix is valid
        row_sums = g.p.sum(axis=1)
        assert not ((~np.isclose(row_sums, 1)) & (~np.isclose(row_sums, 0))).any(), \
            f"Transition matrix rows do not sum to 0 for unfeasible state-action pairs or 1 but to {row_sums}"

        group_history[(total_day-current_day, g.traveler_type)] = copy.deepcopy(g)

    # 7. Convergence
    print("b_star", system.b_star)
    b_star_history[total_day-current_day] = system.b_star
    for g in groups:
        expected_value_vec[current_day, g.traveler_type] = g.expected_value_function
        print(f"Group {g.traveler_type} expected value:", g.expected_value_function)
        err = np.linalg.norm(g.pi - pi_old[:, :, g.traveler_type])
        print(f"Group {g.traveler_type} error:", err)
        error_vec[current_day, g.traveler_type] = err
        if err > threshold:
            converge = False
        

In [ ]:
plot_interactive_policy(
    group_history,
    traveler_type=0,
    u = [0, 1],
    k = range(40),
    t = [6,7,8],
    b = range(40),
    b_star_history=b_star_history
)

# issues : 
# b_star should evolve through days too